# Importing Libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Importing the Dataset

In [2]:
train_data = pd.read_csv("/content/Corona_NLP_train.csv", encoding='latin-1')
test_data = pd.read_csv("/content/Corona_NLP_test.csv", encoding='latin-1')

In [3]:
train_data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [4]:
test_data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,1,44953,NYC,02-03-2020,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative
1,2,44954,"Seattle, WA",02-03-2020,When I couldn't find hand sanitizer at Fred Me...,Positive
2,3,44955,NaN,02-03-2020,Find out how you can protect yourself and love...,Extremely Positive
3,4,44956,Chicagoland,02-03-2020,#Panic buying hits #NewYork City as anxious sh...,Negative
4,5,44957,"Melbourne, Victoria",03-03-2020,#toiletpaper #dunnypaper #coronavirus #coronav...,Neutral


# Preprocessing Dataset

In [5]:
train_data = train_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)
train_data.head(5)

,OriginalTweet,Sentiment
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,advice Talk to your neighbours family to excha...,Positive
2,Coronavirus Australia: Woolworths to give elde...,Positive
3,My food stock is not the only one which is emp...,Positive
4,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [6]:
test_data = test_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)
test_data.head(5)

,OriginalTweet,Sentiment
0,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative
1,When I couldn't find hand sanitizer at Fred Me...,Positive
2,Find out how you can protect yourself and love...,Extremely Positive
3,#Panic buying hits #NewYork City as anxious sh...,Negative
4,#toiletpaper #dunnypaper #coronavirus #coronav...,Neutral


# Data Encoding

In [7]:
le = preprocessing.LabelEncoder()
le.fit(train_data['Sentiment'])
train_data['Sentiment'] = le.transform(train_data['Sentiment'])
test_data['Sentiment'] = le.transform(test_data['Sentiment'])

In [8]:
train_data.head(5)

,OriginalTweet,Sentiment
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,3
1,advice Talk to your neighbours family to excha...,4
2,Coronavirus Australia: Woolworths to give elde...,4
3,My food stock is not the only one which is emp...,4
4,"Me, ready to go at supermarket during the #COV...",0


In [9]:
test_data.head(5)

,OriginalTweet,Sentiment
0,TRENDING: New Yorkers encounter empty supermar...,0
1,When I couldn't find hand sanitizer at Fred Me...,4
2,Find out how you can protect yourself and love...,1
3,#Panic buying hits #NewYork City as anxious sh...,2
4,#toiletpaper #dunnypaper #coronavirus #coronav...,3


In [10]:
X_train = train_data['OriginalTweet']
y_train = train_data['Sentiment']
X_test = test_data['OriginalTweet']
y_test = test_data['Sentiment']

In [11]:
len(X_train), len(X_test)

(41157, 3798)

# Compute text Statistics for Vectorization

In [12]:
averageWordsLength = round(sum([len(i.split()) for i in train_data['OriginalTweet']]) / len(train_data['OriginalTweet']))
totalWordsLength = len(set(" ".join(train_data['OriginalTweet']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 41157
Average words per message: 31
Approximate vocabulary size: 136386


# TextVectorization layer for Tokenization

In [13]:
text_vectorizer = layers.TextVectorization(
    max_tokens=20000,#totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=averageWordsLength
)
text_vectorizer.adapt(X_train)

# Create the Embedding layer

In [14]:
embedding_layer = layers.Embedding(
    input_dim=20000,#totalWordsLength,
    output_dim=128,
    embeddings_initializer='uniform',
)

# Model Architecture

In [15]:
input_layer = layers.Input(shape=(1,), dtype=tf.string)
vectorized_layer = text_vectorizer(input_layer)
embedding_layer = embedding_layer(vectorized_layer)
x = layers.LSTM(64, return_sequences=True)(embedding_layer)
# x = layers.LSTM(64)(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
output_layer = layers.Dense(5, activation='softmax')(x)

model = keras.Model(inputs=input_layer, outputs=output_layer)
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 31)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 31, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 31, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,613,893 (9.97 MB)

 Trainable params: 2,613,893 (9.97 MB)

 Non-trainable params: 0 (0.00 B)

# Training & Evaluating the Model

In [16]:
history = model.fit(X_train.to_numpy(), y_train.to_numpy(), epochs=5, batch_size=128, validation_data=(X_test.to_numpy(), y_test.to_numpy()))

Epoch 1/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 32s 90ms/step - accuracy: 0.3795 - loss: 1.3889 - val_accuracy: 0.6174 - val_loss: 0.9842
Epoch 2/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 28s 86ms/step - accuracy: 0.7130 - loss: 0.7807 - val_accuracy: 0.6417 - val_loss: 0.9284
Epoch 3/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 28s 86ms/step - accuracy: 0.7869 - loss: 0.6084 - val_accuracy: 0.6403 - val_loss: 0.9452
Epoch 4/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.8279 - loss: 0.5088 - val_accuracy: 0.6359 - val_loss: 1.0511
Epoch 5/5
322/322 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.8595 - loss: 0.4226 - val_accuracy: 0.6345 - val_loss: 1.0932


# Sample Prediction